# NCAA Rowing Roster Extraction

This project is a summary of my time as a Division I rower at Duquesne University. Over four years of competing in the Atlantic 10 conference I had many meets and many more teammates. The work here uses various analysis tools (**webscraping, regex, visualizations**) to demonstrate and tell the narrative of my collegiate carrer.

In [7]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

Method to request access to the Duquesne athletics site

In [8]:
def request_access(season_years:str):
    """Make site request for webscraping data

    Args:
        season_years: desired season years in format 20XX-XX

    Returns:
        r: request message (200 indicating success)
    """
    headers = {'user-agent': 'H Valenty personal study (unc6kr@virginia.edu)'}
    r = requests.get(f"https://goduquesne.com/sports/womens-rowing/roster/{season_years}",
                     headers = headers)
    return r

Method to clean front and back end of entries

In [9]:
def remove_str_start_end(s, start, end):
    """string function to clean formatted entries"""
    return s[:start] + s[end + 1:]

Method for primary cleaning of webscraped data

In [10]:
def initial_clean(req):
    """cleaning gathered data to remove
    javascript formatting and code

    Args:
        req: request response of website data

    Returns:
        rep: cleaned data as list
    """
    # access javascript code from request to website
    roster = BeautifulSoup(req.text, 'html').find_all('script', type=True)[0]
    # format javascript as string
    str_roster = str(roster)
    # split string to isolate each rower
    roster_list = str_roster.split("Person")[1:]
    # clean formatted text from each rower
    clean = [a.strip('",\"@context":\"http://schema.org\",') for a in roster_list ]
    # call method to further clean start and end of entries
    rep = [i.strip(remove_str_start_end(clean[0], 31, -9)) for i in clean]

    return rep

Method to transform data into formatted dataframe

In [11]:
def dataframe_format(rep):
    """alter list data to more workable format as dataframe
    Args:
        rep: cleaned data in list format

    Returns:
        duq_roster: pandas dataframe of cleaned data
    """
    # fully split text into list of lists for rower entries
    hold = [rep[i].split(',') for i in range(len(rep))]
    # format into pandas dataframre and selection columns of interest
    duq_roster = pd.DataFrame(hold)[[0,3,4,5]]
    # rename columns 
    replace_map = {0:'photo_url',
              3:'athlete',
              4:'gender',
              5:'page_url'}

    duq_roster = duq_roster.rename(replace_map, axis = 1)
    # drop nulls from dataframe
    duq_roster = duq_roster.drop(duq_roster[duq_roster.photo_url == 'null'].index).reset_index(drop=True)

    return duq_roster

End process cleaning for formatted dataframe

In [12]:
def final_clean(duq_roster):
    """combination of string regex for final cleaning"""
    duq_roster['photo_url'] = [duq_roster.photo_url[i].replace('url":"', '') for i in range(len(duq_roster))]
    duq_roster.athlete = [duq_roster.athlete[i].replace('"name":"', '').strip('"') for i in range(len(duq_roster))]
    duq_roster.gender = [duq_roster.gender[i].replace('"gender":"', '').strip('"') for i in range(len(duq_roster))]
    duq_roster.page_url = [duq_roster.page_url[i].replace('"url":"', '') for i in range(len(duq_roster))]

    return duq_roster

Execute methods for 4 seasons of collegiate career

In [13]:
seasons = ['2020-21','2021-22','2022-23','2023-24']
season_df = {}
# iterate through all 4 seasons to generate data tables
for season in seasons:
    r = request_access(season)
    rep = initial_clean(req=r)
    duq_roster = dataframe_format(rep=rep)
    season_df["season-{0}".format(season)] = final_clean(duq_roster=duq_roster)

Show snippet of 2020-21 season roster

In [19]:
season_df['season-2020-21'].tail(6)

,photo_url,athlete,gender,page_url
51,https://goduquesne.com/images/2020/11/17/Hanna...,Hannah Valenty,F,https://goduquesne.com/roster.aspx?rp_id=10481
52,https://goduquesne.com/images/2019/8/1/New_D_h...,Anna Vignali,F,https://goduquesne.com/roster.aspx?rp_id=10495
53,https://goduquesne.com/images/2020/11/17/Zara_...,Zara Wenzinger,F,https://goduquesne.com/roster.aspx?rp_id=10472
54,https://goduquesne.com/images/2020/11/17/Britt...,Britta Wheeler,F,https://goduquesne.com/roster.aspx?rp_id=10496
55,https://goduquesne.com/images/2019/8/1/New_D_h...,Madelyn Winiarski,F,https://goduquesne.com/roster.aspx?rp_id=10497
56,https://goduquesne.com/images/2020/11/17/Grace...,Grace Yeretzian,F,https://goduquesne.com/roster.aspx?rp_id=10479...


## Stage 2: Getting more rower bio information

Thought process

* get rower web-id and their name to input into web address format
* have to access each rower page for their bio info (class, high school, hometown)
* the personal pages capture for a moment in time the last year the rower was on the team
    * need to access 4 pages where a rower was a senior to get all web-id and names
* once able to automate the process, then can make a new dataframe with all info

Look around a bit

In [20]:
headers = {'user-agent': 'H Valenty personal study (unc6kr@virginia.edu)'}
r = requests.get("https://goduquesne.com/roster.aspx?rp_id=10481",
                     headers = headers)
print(r)

<Response [200]>


Find the web address values for 2023-24 roster members

In [47]:
option = BeautifulSoup(r.text, 'html').find("option", {"value": True})
option

<option value="12128">Abbott, Bridget
	<option value="12129">Abbott, Isabella
	<option value="12130">Ackerman, Kathryn
	<option value="12131">Ardrey, Kelly
	<option value="12269">Benavides, Alice
	<option value="12132">Bosworth, Michaela
	<option value="12161">Boyce, Lauren
	<option value="12162">Brouillard, Rory
	<option value="12250">Casey, Julia
	<option value="12163">Catao, Michelle
	<option value="12270">D’Eramo, Antonina
	<option value="12164">Dahnke, Kiran
	<option value="12261">DeCosmo, Serena
	<option value="12259">DeSaro, Jessica
	<option value="12165">DeStefano, Caitlin
	<option value="12138">Dicke, Cherise
	<option value="12247">Dobrek, Emily
	<option value="12263">Egan, Catherine
	<option value="12195">Elliott, Megan
	<option value="12196">Engel, Paige
	<option value="12245">Ernst, Chloe
	<option value="12256">Foglia, Nora Grace
	<option value="12169">Gallagher, Allyson
	<option value="12266">Gross, Hailey
	<option value="12170">Hesch, Natalie
	<option value="12249">Hinche

### Bio functions to be used once inside a rower profile page

Rower firstname

In [92]:
first_name = BeautifulSoup(r.text, 'html').find("span", {"class": "sidearm-roster-player-first-name"})
rower_fn = str(first_name).split(">")[1].split('<')[0]
rower_fn

'Hannah'

Rower lastname

In [93]:
last_name = BeautifulSoup(r.text, 'html').find("span", {"class": "sidearm-roster-player-last-name"})
rower_ln = str(last_name).split(">")[1].split('<')[0]
rower_ln

'Valenty'

Rower bio information

In [91]:
bio = BeautifulSoup(r.text, 'html').find_all("span", {"class": False, "aria-live": False,
                                                      "data-bind": False})[0:3]

rower_bio = list(map(lambda b: str(b).split(">")[1].split('<')[0], bio))
rower_bio

['Senior', 'Our Lady of the Sacred Heart', 'Carnegie, Pa.']